# 01 — Panel Build

Ingests all 10 verified Tier 1 datasets from `data/raw/` and produces:
- `data/processed/pmsg_snapshots_long.csv` — dated scheme-progress snapshots
- `data/processed/state_covariates_wide.csv` — one row per state, static/annual covariates
- `data/processed/utility_level_tariff.csv` — DISCOM-specific tariff rows kept separate

**Design decisions (see README for full detail):**
- Canonical state list = 36 states/UTs (from the Registrations file)
- Urban/Rural tariff splits are averaged into one state-level figure
- DISCOM-specific rows (Torrent Power, Mumbai BEST/Adani/TATA) are excluded from
  state totals to avoid double-counting, kept separately instead
- Scheme-progress snapshots are kept as separate dated rows — NOT force-merged,
  since source dates are 10+ weeks apart

In [1]:
import pandas as pd
import openpyxl
import sys
from pathlib import Path

sys.path.append("../src")
from state_mapping import normalize_state, CANONICAL_STATES

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

## 1. Scheme-progress snapshots (dated, long format)

In [2]:
snapshot_rows = []

def load_registrations():
    wb = openpyxl.load_workbook(RAW / "Registrations__applications__households_benefited.xlsx", data_only=True)
    ws = wb.active
    date = "2025-02-06"
    source = "Rajya Sabha Session 267, Q931 (ans. 11-Feb-2025)"
    for row in ws.iter_rows(min_row=2, values_only=True):
        if not row[1] or str(row[1]).strip().lower() == "total":
            continue
        state = normalize_state(row[1])
        if state.startswith("__"):
            continue
        for metric, val in [("registrations", row[2]), ("applications", row[3]), ("households_benefited", row[4])]:
            snapshot_rows.append({"state": state, "date": date, "metric": metric,
                                   "value": None if val in (None, "NA") else val, "source": source})

load_registrations()
print(f"Registrations loaded: {len(snapshot_rows)} rows so far")

Registrations loaded: 108 rows so far


In [3]:
def load_installations():
    wb = openpyxl.load_workbook(RAW / "Installations.xlsx", data_only=True)
    ws = wb.active
    date = "2024-11-27"
    source = "Rajya Sabha Session 266, Q945 (ans. 3-Dec-2024)"
    for row in ws.iter_rows(min_row=2, values_only=True):
        if not row[1] or str(row[1]).strip().lower() == "total":
            continue
        state = normalize_state(row[1])
        if state.startswith("__"):
            continue
        val = row[2]
        snapshot_rows.append({"state": state, "date": date, "metric": "installations",
                               "value": None if val in (None, "NA") else val, "source": source})

load_installations()
print(f"Installations loaded: {len(snapshot_rows)} rows so far")

Installations loaded: 143 rows so far


In [4]:
def load_district_maharashtra():
    wb = openpyxl.load_workbook(RAW / "3__District-wise_beneficiaries__Maharashtra__11-12-2024.xlsx", data_only=True)
    ws = wb.active
    date = "2024-12-11"
    source = "Rajya Sabha Session 266, Q2545 (district-level, Maharashtra only)"
    total = 0
    for row in ws.iter_rows(min_row=2, values_only=True):
        if not row[1] or str(row[1]).strip().lower() == "total":
            continue
        val = row[2]
        if isinstance(val, (int, float)):
            total += val
    snapshot_rows.append({"state": "Maharashtra", "date": date, "metric": "cfa_beneficiaries_district_sum",
                           "value": total, "source": source})

load_district_maharashtra()
print(f"District (Maharashtra) loaded: {len(snapshot_rows)} rows so far")

District (Maharashtra) loaded: 144 rows so far


In [5]:
def load_district_discom_conversion():
    wb = openpyxl.load_workbook(RAW / "District_and_discom_info-PM_surya_ghar.xlsx", data_only=True)
    ws = wb["Discom info"]
    date = "2026-08-23"
    source = "Live PMSG portal dashboard (Discom info tab, as on 23-Aug-2026)"
    agg = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        state_raw = row[0]
        if not state_raw:
            continue
        state = normalize_state(state_raw)
        if state.startswith("__"):
            continue
        app, inst, hh = row[2], row[3], row[4]
        d = agg.setdefault(state, {"applications": 0, "installations": 0, "households": 0})
        if isinstance(app, (int, float)):
            d["applications"] += app
        if isinstance(inst, (int, float)):
            d["installations"] += inst
        if isinstance(hh, (int, float)):
            d["households"] += hh
    for state, d in agg.items():
        for metric_key, col in [("applications", "applications"), ("installations", "installations"),
                                 ("households_benefited", "households")]:
            snapshot_rows.append({"state": state, "date": date, "metric": f"discom_{metric_key}",
                                   "value": d[col], "source": source})
        if d["applications"] > 0:
            ratio = d["installations"] / d["applications"]
            snapshot_rows.append({"state": state, "date": date, "metric": "conversion_ratio",
                                   "value": round(ratio, 4), "source": source})

load_district_discom_conversion()
print(f"District/Discom conversion loaded: {len(snapshot_rows)} rows total")

District/Discom conversion loaded: 284 rows total


In [6]:
snapshots_df = pd.DataFrame(snapshot_rows).sort_values(["state", "date", "metric"]).reset_index(drop=True)

# Sanity check: any unmapped state names slipped through?
unmapped = snapshots_df[snapshots_df["state"].astype(str).str.startswith("__UNMAPPED__")]
print("Unmapped states found:", unmapped["state"].unique() if len(unmapped) else "None — all clean")

snapshots_df.to_csv(PROCESSED / "pmsg_snapshots_long.csv", index=False)
snapshots_df.head(10)

Unmapped states found: None — all clean


,state,date,metric,value,source
0,Andaman and Nicobar Islands,2025-02-06,applications,1.250000e+02,"Rajya Sabha Session 267, Q931 (ans. 11-Feb-2025)"
1,Andaman and Nicobar Islands,2025-02-06,households_benefited,5.000000e+00,"Rajya Sabha Session 267, Q931 (ans. 11-Feb-2025)"
2,Andaman and Nicobar Islands,2025-02-06,registrations,1.132000e+03,"Rajya Sabha Session 267, Q931 (ans. 11-Feb-2025)"
3,Andaman and Nicobar Islands,2026-08-23,conversion_ratio,6.617000e-01,"Live PMSG portal dashboard (Discom info tab, a..."
4,Andaman and Nicobar Islands,2026-08-23,discom_applications,6.710000e+02,"Live PMSG portal dashboard (Discom info tab, a..."
5,Andaman and Nicobar Islands,2026-08-23,discom_households_benefited,4.630000e+02,"Live PMSG portal dashboard (Discom info tab, a..."
6,Andaman and Nicobar Islands,2026-08-23,discom_installations,4.440000e+02,"Live PMSG portal dashboard (Discom info tab, a..."
7,Andhra Pradesh,2024-11-27,installations,6.725000e+03,"Rajya Sabha Session 266, Q945 (ans. 3-Dec-2024)"
8,Andhra Pradesh,2025-02-06,applications,1.107234e+06,"Rajya Sabha Session 267, Q931 (ans. 11-Feb-2025)"
9,Andhra Pradesh,2025-02-06,households_benefited,1.203600e+04,"Rajya Sabha Session 267, Q931 (ans. 11-Feb-2025)"


## 2. Static / annual state-level covariates

In [7]:
covariates = {s: {} for s in CANONICAL_STATES}
utility_tariff_rows = []

def add_cov(state, key, val):
    if state in covariates:
        covariates[state][key] = val

In [8]:
# --- Tariff (Domestic 4KW), 2024 & 2025 ---
def load_tariff(path, year_label):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["Domestic - 4KW"]
    grouped = {}
    for row in ws.iter_rows(min_row=2, values_only=True):
        raw_state, util, date, avg, duty, total = row[:6]
        if not util:
            continue
        state = normalize_state(util)
        if state.startswith("__UTILITY"):
            utility_tariff_rows.append({"utility": util, "year": year_label, "total_tariff_paise_per_kwh": total})
            continue
        if state.startswith("__") or state not in covariates:
            continue
        grouped.setdefault(state, []).append(total)
    for state, vals in grouped.items():
        vals = [v for v in vals if isinstance(v, (int, float))]
        if vals:
            add_cov(state, f"tariff_domestic4kw_{year_label}_paise_per_kwh", round(sum(vals) / len(vals), 2))

load_tariff(RAW / "PMSG_Tariff_-_2024.xlsx", "2024")
load_tariff(RAW / "PMSG_Tariff_-_2025.xlsx", "2025")
print("Tariff loaded for", sum(1 for s in covariates if "tariff_domestic4kw_2025_paise_per_kwh" in covariates[s]), "states")

Tariff loaded for 35 states


In [9]:
# --- Solar potential (both wasteland scenarios) ---
wb = openpyxl.load_workbook(RAW / "Solar_potential.xlsx", data_only=True)
ws = wb.active
for row in ws.iter_rows(min_row=2, values_only=True):
    raw_state = row[0]
    if not raw_state or str(raw_state).strip().lower() in ("total", "others"):
        continue
    state = normalize_state(raw_state)
    if state.startswith("__") or state not in covariates:
        continue
    add_cov(state, "solar_potential_6.69pct_wasteland_gw", row[2] if isinstance(row[2], (int, float)) else None)
    v3 = row[3]
    add_cov(state, "solar_potential_3pct_wasteland_gw", v3 if isinstance(v3, (int, float)) else None)

print("Solar potential loaded")

Solar potential loaded


In [10]:
# --- ToD regime (domestic applicability flag) ---
wb = openpyxl.load_workbook(RAW / "PMSG_ToD_Regime_by_State.xlsx", data_only=True)
ws = wb.active
for row in ws.iter_rows(min_row=5, values_only=True):
    raw_state = row[0]
    if not raw_state:
        continue
    state = normalize_state(raw_state)
    if state.startswith("__") or state not in covariates:
        continue
    add_cov(state, "tod_domestic_applicable", row[3])

print("ToD regime loaded")

ToD regime loaded


In [11]:
# --- Solar irradiance (NASA POWER climatology, annual) ---
# NOTE: source file had "Telengana" misspelled — state_mapping.py already
# includes this as a known variant, so it maps correctly here.
wb = openpyxl.load_workbook(RAW / "States_nasa.xlsx", data_only=True)
ws = wb.active
for row in ws.iter_rows(values_only=True):
    if not row or not row[0] or "," not in str(row[0]):
        continue
    label = str(row[0])
    try:
        _, state_part = label.split(",", 1)
    except ValueError:
        continue
    state = normalize_state(state_part)
    if state.startswith("__") or state not in covariates:
        continue
    ann = row[13] if len(row) > 13 else None
    if isinstance(ann, (int, float)):
        add_cov(state, "irradiance_annual_kwh_m2_day", round(ann, 4))

print("Irradiance loaded for", sum(1 for s in covariates if "irradiance_annual_kwh_m2_day" in covariates[s]), "states")

Irradiance loaded for 28 states


In [12]:
# --- MNRE RE capacity: two-point (2018 vs 2025) ---
wb = openpyxl.load_workbook(RAW / "state-wise_RE_and_non-RE_installed_capacity_at_two_points.xlsx", data_only=True)
ws = wb["Sheet1"]
for row in ws.iter_rows(min_row=2, values_only=True):
    raw_state = row[1] if len(row) > 1 else None
    if not raw_state or "INDIA" in str(raw_state).upper():
        continue
    state = normalize_state(raw_state)
    if state.startswith("__") or state not in covariates:
        continue
    nonre18, re18, nonre25, re25 = row[2], row[3], row[4], row[5]
    add_cov(state, "mnre_nonre_capacity_2018_mw", nonre18)
    add_cov(state, "mnre_re_capacity_2018_mw", re18)
    add_cov(state, "mnre_nonre_capacity_2025_mw", nonre25)
    add_cov(state, "mnre_re_capacity_2025_mw", re25)

print("MNRE RE capacity loaded")

MNRE RE capacity loaded


In [13]:
cov_df = pd.DataFrame.from_dict(covariates, orient="index")
cov_df.index.name = "state"
cov_df = cov_df.reset_index()

cov_df.to_csv(PROCESSED / "state_covariates_wide.csv", index=False)

util_df = pd.DataFrame(utility_tariff_rows)
util_df.to_csv(PROCESSED / "utility_level_tariff.csv", index=False)

print("Covariate table shape:", cov_df.shape)
print("\nMissing values per column:")
print(cov_df.isna().sum())
cov_df.head(10)

Covariate table shape: (36, 11)

Missing values per column:
state                                    0
tariff_domestic4kw_2024_paise_per_kwh    1
tariff_domestic4kw_2025_paise_per_kwh    1
solar_potential_6.69pct_wasteland_gw     1
solar_potential_3pct_wasteland_gw        5
tod_domestic_applicable                  5
irradiance_annual_kwh_m2_day             8
mnre_nonre_capacity_2018_mw              0
mnre_re_capacity_2018_mw                 0
mnre_nonre_capacity_2025_mw              0
mnre_re_capacity_2025_mw                 0
dtype: int64


,state,tariff_domestic4kw_2024_paise_per_kwh,tariff_domestic4kw_2025_paise_per_kwh,solar_potential_6.69pct_wasteland_gw,solar_potential_3pct_wasteland_gw,tod_domestic_applicable,irradiance_annual_kwh_m2_day,mnre_nonre_capacity_2018_mw,mnre_re_capacity_2018_mw,mnre_nonre_capacity_2025_mw,mnre_re_capacity_2025_mw
0,Andhra Pradesh,653.0,653.0,223.28,38.44,Partial — LT-I(A) domestic gets a rebate only ...,5.1230,16507.20,8060.58,17905.34,12114.48
1,Arunachal Pradesh,400.0,400.0,0.47,8.65,No,3.6487,0.00,517.00,0.00,1270.46
2,Assam,805.0,941.0,19.17,13.76,No,4.3882,1103.95,378.39,1347.36,582.65
3,Bihar,952.0,936.0,32.99,11.20,No,4.7798,5480.00,346.18,9060.00,539.26
4,Chhattisgarh,535.0,557.0,126.48,18.27,No,5.1634,22968.00,785.42,23688.00,1828.46
5,Goa,341.0,373.0,6.75,0.88,No,5.3167,48.00,1.08,48.00,58.43
6,Gujarat,469.0,469.0,243.22,35.77,No,5.5255,23483.41,9339.45,24828.41,33393.03
7,Haryana,481.0,490.0,6.47,4.56,No,5.2246,5971.59,510.49,5761.59,2449.94
8,Himachal Pradesh,551.0,643.0,21.50,33.84,No,4.7004,0.00,10514.30,0.00,12196.19
9,Jammu and Kashmir,323.0,323.0,8.59,111.05,No,NaN,175.00,3650.22,175.00,3624.42


## 3. Output

Two files now sit in `data/processed/`:
- `pmsg_snapshots_long.csv`
- `state_covariates_wide.csv`
- `utility_level_tariff.csv`

Next: open `02_eda.ipynb` to explore this panel.